# Praktikum Machine Learning — Bab 7

## Evaluasi Model: Confusion Matrix, Precision-Recall, ROC-AUC, dan Stratified K-Fold CV

| | |
|---|---|
| **Nama** | Ahmad Dandi Subhani |
| **NPM** | 202343500126 |
| **Kelas** | R7B |
| **Mata Kuliah** | Machine Learning |
| **Dosen** | Nurfidah Dwitiyanti, M.Si. |

> Notebook ini dikerjakan di **Google Colab / Jupyter Notebook**. Jalankan setiap cell **secara berurutan** dari atas ke bawah (`random_state=42` dipakai di seluruh notebook agar hasilnya dapat direproduksi).

## Ringkasan Konsep

**Confusion matrix** adalah tabel 2×2 (untuk biner) yang membandingkan label aktual dengan label prediksi:

| | Diprediksi Negatif | Diprediksi Positif |
|---|---|---|
| **Aktual Negatif** | TN (True Negative): benar menebak negatif | FP (False Positive): salah menebak positif (*kesalahan tipe I*) |
| **Aktual Positif** | FN (False Negative): salah menebak negatif (*kesalahan tipe II*) | TP (True Positive): benar menebak positif |

**Rumus metrik (dihitung dari confusion matrix):**
- **Accuracy** = (TP + TN) / (TP + TN + FP + FN) — proporsi prediksi yang benar.
- **Precision** = TP / (TP + FP) — dari semua yang diprediksi positif, berapa yang benar-benar positif.
- **Recall** = TP / (TP + FN) — dari semua yang benar-benar positif, berapa yang berhasil ditangkap (*sensitivity/TPR*).
- **F1-score** = 2 · (Precision · Recall) / (Precision + Recall) — rata-rata harmonik precision & recall (menghukum nilai yang timpang).

**Averaging untuk multiclass** (hitung metrik per kelas, lalu gabungkan):
- **Macro**: rata-rata biasa dari tiap kelas — semua kelas bobotnya sama (kelas minoritas tetap diperhitungkan).
- **Micro**: jumlahkan dulu seluruh TP/FP/FN lintas kelas, baru hitung metrik — kelas besar mendominasi.
- **Weighted**: rata-rata metrik per kelas dibobot jumlah sampel tiap kelas (*support*).

**Kurva ROC & AUC:** plot **TPR** (recall) vs **FPR** = FP/(FP+TN) pada berbagai threshold. **AUC** (0–1) = luas di bawah kurva: 1.0 = sempurna, 0.5 = setara tebakan acak. Kelebihan: tidak bergantung satu threshold. Kekurangan: bisa terlihat "bagus" pada data imbalance karena banyaknya TN mudah.

**Kurva Precision-Recall & PR-AUC** (*average precision*): plot precision vs recall pada berbagai threshold. Baseline model acak = prevalensi kelas positif (bukan 0.5!), sehingga **PR-AUC lebih jujur untuk kelas positif yang langka** (mis. penipuan, kanker).

**Stratified k-fold CV:** data dibagi k lipatan dengan **proporsi kelas tiap lipatan dijaga sama** dengan data asli; tiap lipatan bergantian jadi data uji. Jauh lebih andal daripada satu kali split — terutama untuk data imbalance, di mana K-Fold biasa bisa menghasilkan lipatan tanpa satu pun sampel kelas minoritas. Dilaporkan sebagai skor tiap fold + **mean ± std**.

**Kapan memakai metrik apa:**

| Situasi | Metrik yang tepat | Kenapa |
|---|---|---|
| Kelas seimbang, semua error setara | Accuracy / F1 | Cukup mewakili performa keseluruhan |
| FN mahal (kanker, penipuan) | Recall tinggi | Jangan sampai kasus positif lolos |
| FP mahal (spam filter, blokir akun) | Precision tinggi | Jangan sampai salah menuduh |
| Kelas positif langka | PR-AUC, F1, bukan accuracy | Accuracy bisa 99% hanya dengan selalu menebak negatif |
| Membandingkan model secara adil | Stratified k-fold CV (mean ± std) | Satu split bisa "kebetulan bagus/jelek" |

## 7.12 Praktikum — Implementasi Python

Kita memakai **dataset real** `breast_cancer` dari scikit-learn (569 sampel, 30 fitur) dengan model `LogisticRegression` di dalam `Pipeline(StandardScaler(), ...)` agar fitur yang skalanya berbeda-beda tidak menghambat konvergensi (`max_iter=2000`).

### Praktikum 7.1 — Persiapan Library

Pertama, kita memuat semua library yang dibutuhkan. Cell ini tidak menghasilkan output selain "selesai" — tidak adanya error berarti semua library tersedia.

In [ ]:
%matplotlib inline
# Menjalankan di Google Colab / Jupyter Notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_curve, auc, roc_auc_score, precision_recall_curve, average_precision_score)

print("Semua library berhasil dimuat.")

**Penjelasan output:** hanya tercetak `Semua library berhasil dimuat.` tanpa error — artinya `pandas`, `numpy`, `matplotlib`, `seaborn`, dan `scikit-learn` semuanya tersedia dan siap dipakai.

### Praktikum 7.2 — Dataset & Pembagian Data

Dataset `breast_cancer` memuat ciri-ciri tumor payudara dengan label `0 = ganas (malignant)` dan `1 = jinak (benign)`. Kita bagi dengan `stratify=y` agar proporsi kelas di data latih dan uji sama dengan data aslinya.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print("Bentuk data:", X.shape, "| jumlah fitur:", len(cancer.feature_names))
print("Distribusi kelas -> ganas (0):", (y == 0).sum(), "| jinak (1):", (y == 1).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("\nData latih:", X_train.shape, "| Data uji:", X_test.shape)
print("Proporsi jinak — data asli: %.3f | latih: %.3f | uji: %.3f"
      % (y.mean(), y_train.mean(), y_test.mean()))

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, random_state=42))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Model selesai dilatih. Akurasi data uji: %.4f" % model.score(X_test, y_test))

**Penjelasan output:**
1. Dataset berukuran **(569, 30)** — 569 pasien dengan 30 fitur pengukuran tumor; kelasnya **212 ganas (0)** dan **357 jinak (1)**, jadi tidak seimbang sempurna (62,7% jinak).
2. Berkat `stratify=y`, proporsi kelas jinak tetap **0,627** di data latih maupun uji — pembagiannya adil.
3. Model `Pipeline(StandardScaler(), LogisticRegression)` berhasil dilatih dengan **akurasi data uji 0,9860** — angka awal yang sangat baik, tetapi akurasi saja belum cukup; kita bedah lebih dalam di bawah.

### Praktikum 7.3(a) — Confusion Matrix (Heatmap)

Confusion matrix menunjukkan *di mana* model salah. Kita visualkan sebagai heatmap dengan label Bahasa Indonesia.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
print("TN = %d | FP = %d | FN = %d | TP = %d" % (TN, FP, FN, TP))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Diprediksi: Ganas (0)", "Diprediksi: Jinak (1)"],
            yticklabels=["Aktual: Ganas (0)", "Aktual: Jinak (1)"])
plt.title("Confusion Matrix — Prediksi Tumor (Logistic Regression)")
plt.xlabel("Hasil prediksi model")
plt.ylabel("Label sebenarnya")
plt.tight_layout()
plt.show()

**Penjelasan output:**
1. Dari 143 data uji: **TN = 52** (ganas terdeteksi benar), **TP = 89** (jinak terdeteksi benar), **FP = 1** (jinak dikira ganas), **FN = 1** (ganas dikira jinak).
2. Total benar = 141 dari 143 → akurasi manual = 141/143 = **0,9860**, cocok dengan skor model.
3. Hanya ada **2 kesalahan** dan keduanya "simetris" (satu FP, satu FN) — model tidak bias ke salah satu arah. Dalam konteks medis, FN = 1 adalah yang paling perlu diwaspadai karena satu kasus ganas lolos sebagai jinak.

### Praktikum 7.3(b) — Classification Report

`classification_report` merangkum precision, recall, dan F1 per kelas plus rata-ratanya.

In [ ]:
print(classification_report(y_test, y_pred, target_names=["ganas (0)", "jinak (1)"]))

**Penjelasan output** (membaca tiap angka):
1. **Kelas ganas (0), support = 53:** precision **0,98** = dari semua yang diprediksi ganas, 98% memang ganas; recall **0,98** = dari 53 kasus ganas asli, 52 tertangkap (hanya 1 lolos sebagai FN); F1 **0,98**.
2. **Kelas jinak (1), support = 90:** precision **0,99**, recall **0,99**, F1 **0,99** — sedikit lebih tinggi karena kelas ini lebih banyak (mayoritas).
3. **Accuracy 0,99** (dibulatkan dari 0,9860) = proporsi prediksi benar secara keseluruhan.
4. **Macro avg 0,99** (rata-rata kedua kelas dengan bobot sama) dan **weighted avg 0,99** (dibobot 53 vs 90) hampir identik — wajar karena performa kedua kelas sama-sama tinggi; perbedaannya baru terasa kalau salah satu kelas jauh lebih buruk.

### Praktikum 7.3(c) — Kurva ROC & AUC

Kurva ROC dibuat dari probabilitas prediksi (`predict_proba`), bukan dari satu threshold 0,5 — sehingga ia menilai kualitas ranking model secara keseluruhan.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label="Logistic Regression (AUC = %.4f)" % roc_auc, linewidth=2)
plt.plot([0, 1], [0, 1], "k--", label="Tebakan acak (AUC = 0.50)")
plt.title("Kurva ROC — Prediksi Tumor")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate / Recall (TPR)")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Nilai AUC: %.4f" % roc_auc)

**Penjelasan output:**
1. **AUC = 0,9977** — sangat dekat dengan 1,0 (sempurna). Kurvanya menempel di sudut kiri atas, artinya model memisahkan kelas ganas dan jinak dengan hampir sempurna di hampir semua threshold.
2. Garis putus-putus diagonal adalah baseline tebakan acak (AUC = 0,50) — model kita jauh di atasnya.
3. Kelebihan: satu angka merangkum performa di *semua* threshold. Kekurangan (akan kita buktikan di Eksperimen C): pada data yang timpang, AUC bisa tetap tinggi karena banyaknya true negative yang "mudah".

### Praktikum 7.3(d) — Kurva Precision-Recall & PR-AUC

Kurva PR fokus pada kelas positif (jinak): bagaimana precision berubah saat kita menaikkan recall.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
baseline = y_test.mean()

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label="Logistic Regression (PR-AUC = %.4f)" % pr_auc, linewidth=2)
plt.axhline(baseline, color="k", linestyle="--",
            label="Baseline acak (prevalensi = %.3f)" % baseline)
plt.title("Kurva Precision-Recall — Prediksi Tumor")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("PR-AUC (average precision): %.4f" % pr_auc)

**Penjelasan output:**
1. **PR-AUC = 0,9986** — hampir sempurna: model mempertahankan precision tinggi bahkan saat recall dinaikkan mendekati 1,0.
2. Garis baseline acak berada di **0,627** (= prevalensi kelas positif), bukan 0,5 — ini ciri khas kurva PR: baseline-nya mengikuti proporsi kelas positif.
3. Karena data ini relatif seimbang, PR-AUC (0,9986) dan ROC-AUC (0,9977) hampir sama. Perbedaannya baru dramatis pada data yang timpang — lihat Eksperimen C.

### Praktikum 7.3(e) — Stratified K-Fold Cross-Validation

Satu kali split bisa "kebetulan bagus". Dengan `StratifiedKFold(n_splits=5)`, seluruh data dipakai sebagai data uji tepat satu kali, dengan proporsi kelas yang selalu terjaga di tiap lipatan.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skor = cross_val_score(model, X, y, cv=skf, scoring="accuracy")
skor_f1 = cross_val_score(model, X, y, cv=skf, scoring="f1")

print("Akurasi tiap fold :", np.round(skor, 4))
print("F1 tiap fold      :", np.round(skor_f1, 4))
print("\nAkurasi: %.4f ± %.4f" % (skor.mean(), skor.std()))
print("F1     : %.4f ± %.4f" % (skor_f1.mean(), skor_f1.std()))

fold = 1
for tr_idx, te_idx in skf.split(X, y):
    print("Fold %d: proporsi jinak di uji = %.4f" % (fold, y[te_idx].mean()))
    fold += 1

**Penjelasan output:**
1. Akurasi tiap fold = **[0,9737; 0,9474; 0,9649; 0,9912; 0,9912]** → rata-rata **0,9737 ± 0,0166**; F1 = **0,9794 ± 0,0127**.
2. Angka CV (0,9737) sedikit *lebih rendah* dari akurasi satu split tadi (0,9860) — ini normal dan justru lebih jujur: satu split bisa kebetulan "mudah", sedangkan CV merata-ratakan 5 lipatan.
3. Proporsi kelas jinak di tiap lipatan uji selalu **≈ 0,623–0,632** (hampir sama dengan 0,627 data asli) — inilah gunanya *stratified*: tidak ada lipatan yang kekurangan salah satu kelas.
4. Simpulan: model stabil (std kecil, 0,0166) dan generalisasinya bagus — bukan sekadar hafal satu split tertentu.

### Praktikum 7.3(f) — Demo Averaging Multiclass pada Dataset Digits

Dataset `digits` punya **10 kelas** (angka 0–9). Kita lihat bagaimana macro, micro, dan weighted F1 dihitung — modelnya tetap LogisticRegression.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(Xd, yd, test_size=0.25, random_state=42, stratify=yd)

lr_multi = LogisticRegression(max_iter=3000, random_state=42)
lr_multi.fit(Xtr_d, ytr_d)
y_pred_d = lr_multi.predict(Xte_d)

print("F1 macro    : %.4f" % f1_score(yte_d, y_pred_d, average="macro"))
print("F1 micro    : %.4f" % f1_score(yte_d, y_pred_d, average="micro"))
print("F1 weighted : %.4f" % f1_score(yte_d, y_pred_d, average="weighted"))
print()
print(classification_report(yte_d, y_pred_d))

**Penjelasan output:**
1. F1 **macro = 0,9619**, **micro = 0,9622**, **weighted = 0,9623** — ketiganya hampir sama karena tiap digit punya support yang seimbang (43–46 sampel).
2. **Micro F1** pada multiclass selalu sama dengan accuracy (0,9622) — karena micro menjumlahkan TP/FP/FN global, ia identik dengan proporsi prediksi benar.
3. Kelas tersulit adalah **digit 8** (F1 = 0,86; precision dan recall sama-sama 0,86) — wajar, bentuk "8" memang paling mirip dengan digit lain (mis. tertukar dengan 1 atau 9). Kelas termudah: digit 0 dan 7 (F1 ≈ 0,99–1,00).
4. Pelajaran: kalau datanya seimbang, pilihan averaging tidak terlalu berpengaruh; averaging baru "berbicara" saat ada kelas minoritas yang performanya jeblok — lihat Latihan B.

## Eksperimen (Coba-Coba)

Tiga percobaan untuk menguji pemahaman: (A) efek threshold keputusan, (B) stabilitas evaluasi, (C) kejujuran metrik pada data timpang.

### Eksperimen A — Mengubah Threshold Keputusan (0,5 → 0,3 dan 0,7)

Secara default `predict()` memakai threshold 0,5. Dengan `predict_proba`, kita bisa menggesernya: threshold rendah = "gampang" memprediksi positif (recall naik, precision turun), threshold tinggi = sebaliknya.

In [ ]:
print(f"{'Threshold':<10} {'Accuracy':<9} {'Precision':<10} {'Recall':<8} {'F1':<8}")
print("-" * 50)
hasil_thr = []
for thr in [0.3, 0.5, 0.7]:
    y_thr = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_test, y_thr)
    prec = precision_score(y_test, y_thr)
    rec = recall_score(y_test, y_thr)
    f1 = f1_score(y_test, y_thr)
    hasil_thr.append((thr, prec, rec, f1))
    print(f"{thr:<10} {acc:<9.4f} {prec:<10.4f} {rec:<8.4f} {f1:<8.4f}")

plt.figure(figsize=(6, 4.5))
ths = [h[0] for h in hasil_thr]
plt.plot(ths, [h[1] for h in hasil_thr], "o-", label="Precision")
plt.plot(ths, [h[2] for h in hasil_thr], "s-", label="Recall")
plt.plot(ths, [h[3] for h in hasil_thr], "^-", label="F1")
plt.title("Trade-off Precision vs Recall terhadap Threshold")
plt.xlabel("Threshold keputusan")
plt.ylabel("Skor")
plt.xticks(ths)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Penjelasan output & interpretasi:**
1. **Threshold 0,3:** recall = **1,0000** (semua kasus jinak tertangkap, nol FN!) tetapi precision turun ke **0,9677** — ada beberapa ganas yang salah dikira jinak. Cocok untuk *skrining awal*: lebih baik "salah alarm" daripada melewatkan kasus.
2. **Threshold 0,5 (default):** precision = recall = **0,9889**, F1 tertinggi (**0,9889**) — titik seimbang.
3. **Threshold 0,7:** precision naik sedikit ke **0,9882**, tetapi recall anjlok ke **0,9333** (6 kasus jinak lolos) dan akurasi turun ke 0,9510. Cocok kalau false positive sangat mahal (mis. tindakan medis invasif), tetapi berbahaya untuk skrining.
4. **Pelajaran:** tidak ada threshold "terbaik" secara absolut — pilihannya tergantung **biaya kesalahan** (FN vs FP), dan inilah alasan kurva ROC/PR digambar di *semua* threshold, bukan satu titik.

### Eksperimen B — Split Biasa vs Stratified 5-Fold: Seberapa Stabil?

Model yang sama dievaluasi dua cara: (1) sepuluh kali `train_test_split` **tanpa** `stratify` (random_state 1–10), dan (2) stratified 5-fold CV. Seberapa stabil skornya?

In [ ]:
accs, f1s, props = [], [], []
for rs in range(1, 11):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=rs)  # tanpa stratify
    m = Pipeline([("scaler", StandardScaler()),
                  ("lr", LogisticRegression(max_iter=2000, random_state=42))])
    m.fit(Xtr, ytr)
    yp = m.predict(Xte)
    accs.append(accuracy_score(yte, yp))
    f1s.append(f1_score(yte, yp))
    props.append(yte.mean())
accs, f1s, props = map(np.array, (accs, f1s, props))

print("Split biasa (10x, tanpa stratify):")
print("  Akurasi: mean = %.4f, std = %.4f, rentang = [%.4f, %.4f]"
      % (accs.mean(), accs.std(), accs.min(), accs.max()))
print("  Proporsi jinak di data uji: rentang = [%.3f, %.3f] (asli: %.3f)"
      % (props.min(), props.max(), y.mean()))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sa = cross_val_score(model, X, y, cv=skf, scoring="accuracy")
sf = cross_val_score(model, X, y, cv=skf, scoring="f1")
print("\nStratified 5-fold CV:")
print("  Akurasi: mean = %.4f, std = %.4f" % (sa.mean(), sa.std()))
print("  F1     : mean = %.4f, std = %.4f" % (sf.mean(), sf.std()))

**Penjelasan output & interpretasi:**
1. Sepuluh split biasa memberi akurasi rata-rata **0,9741 ± 0,0094** dengan rentang **0,958–0,986** — sedangkan stratified 5-fold memberi **0,9737 ± 0,0166**. Rata-ratanya hampir sama, tetapi *satu* split tunggal bisa "kebetulan" setinggi 0,986 (seperti di Praktikum 7.2!) — lebih tinggi dari estimasi CV yang jujur.
2. Tanpa `stratify`, proporsi kelas jinak di data uji **bergeser antara 0,538–0,685** (asli 0,627) — evaluasi bisa bias karena komposisi uji berubah-ubah. Pada stratified CV proporsinya selalu terjaga (≈ 0,623–0,632).
3. Std CV (0,0166) tampak sedikit lebih besar, tetapi itu std yang *jujur*: tiap fold hanya 114 sampel uji dan **seluruh 569 data** dipakai sebagai uji tepat sekali — tidak ada data yang "disembunyikan" dari evaluasi.
4. **Pelajaran:** jangan percaya satu angka dari satu split. Stratified k-fold memberi estimasi generalisasi yang lebih dapat diandalkan karena bebas dari "keberuntungan split" dan proporsi kelas yang bergeser.

### Eksperimen C — ROC-AUC vs PR-AUC pada Data Imbalance Buatan

Kita buat data timpang artifisial: seluruh 212 sampel ganas dipertahankan, tetapi kelas jinak (positif) disusutkan menjadi hanya 40 sampel (15,9%). Metrik mana yang lebih jujur?

In [ ]:
rng = np.random.RandomState(42)
idx_jinak = rng.choice(np.where(y == 1)[0], size=40, replace=False)
idx_ganas = np.where(y == 0)[0]
X_imb = np.vstack([X[idx_ganas], X[idx_jinak]])
y_imb = np.concatenate([np.zeros(len(idx_ganas)), np.ones(len(idx_jinak))])
print("Distribusi baru -> ganas (0): %d | jinak/positif (1): %d (%.1f%%)"
      % ((y_imb == 0).sum(), (y_imb == 1).sum(), 100 * y_imb.mean()))

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)
m_imb = Pipeline([("scaler", StandardScaler()),
                  ("lr", LogisticRegression(max_iter=2000, random_state=42))])
m_imb.fit(Xtr_i, ytr_i)
yp_i = m_imb.predict(Xte_i)
yprob_i = m_imb.predict_proba(Xte_i)[:, 1]

dummy_acc = (yte_i == 0).mean()  # model "malas": selalu menebak negatif
print("\nAkurasi model      : %.4f" % accuracy_score(yte_i, yp_i))
print("Akurasi model malas: %.4f  (selalu menebak negatif)" % dummy_acc)
print("ROC-AUC            : %.4f" % roc_auc_score(yte_i, yprob_i))
print("PR-AUC             : %.4f  (baseline acak = %.3f)" % (
    average_precision_score(yte_i, yprob_i), yte_i.mean()))
tn, fp, fn, tp = confusion_matrix(yte_i, yp_i).ravel()
print("Confusion matrix: TN=%d FP=%d FN=%d TP=%d" % (tn, fp, fn, tp))
print("Precision kelas positif = %d/%d = %.3f | Recall kelas positif = %d/%d = %.3f"
      % (tp, tp + fp, tp / (tp + fp), tp, tp + fn, tp / (tp + fn)))

**Penjelasan output & interpretasi:**
1. **Accuracy menyesatkan:** model "malas" yang *selalu menebak negatif* (tidak menangkap satu pun kasus positif!) sudah mendapat akurasi **0,8421** — hanya 13 poin di bawah model asli (0,9737). Kalau hanya melihat accuracy, model malas terlihat "lumayan".
2. **ROC-AUC = 0,9844** tetap terlihat fantastis — karena ROC dihitung dari FPR yang penyebutnya didominasi 64 true negative yang "mudah". Ia *optimis* pada data timpang.
3. **PR-AUC = 0,9464** dengan baseline acak hanya **0,158**: angka ini memang lebih rendah dari ROC-AUC dan sepenuhnya fokus pada kelas positif yang langka — precision positif = 10/10 = **1,000** tetapi recall positif = 10/12 = **0,833** (2 kasus positif lolos). Inilah cerita yang sebenarnya penting.
4. **Pelajaran:** pada data imbalance, **PR-AUC (dan precision/recall/F1 kelas minoritas) jauh lebih jujur** daripada accuracy maupun ROC-AUC. Inilah fondasi argumen studi kasus di bawah.

## Studi Kasus — Deteksi Penipuan Transaksi Kartu Kredit

**Skenario:** sebuah bank ingin mendeteksi transaksi penipuan (*fraud*). Dari 100.000 transaksi, hanya sekitar **100 yang fraud (0,1%)** — kelas positif sangat langka, dan setiap fraud yang lolos merugikan nasabah.

**Kenapa akurasi saja menyesatkan?**

| Model | Akurasi | Fraud tertangkap (recall) |
|---|---|---|
| Model "malas": selalu menebak *bukan fraud* | **99,9%** | **0%** (semua 100 fraud lolos!) |
| Model ML yang "hanya" 99,0% akurat tapi recall 80% | 99,0% | **80%** (80 fraud tertangkap) |

Model malas menang di akurasi (99,9% > 99,0%) tetapi **total gagal** pada tugas sebenarnya. Ini persis yang ditunjukkan Eksperimen C: model malas mendapat 84% akurasi tanpa menangkap satu pun kasus positif.

**Metrik yang harus dipakai dan kenapa:**

| Metrik | Peran dalam kasus fraud |
|---|---|
| **Recall kelas fraud** | Ukuran utama: berapa persen fraud yang berhasil dicegah. FN = uang nasabah hilang. |
| **Precision kelas fraud** | Biaya operasional: FP = transaksi sah diblokir → nasabah komplain. Trade-off-nya diatur lewat **threshold** (Eksperimen A): turunkan threshold untuk mengejar recall saat rugi fraud lebih mahal daripada komplain. |
| **F1 / PR-AUC** | Satu angka ringkas yang jujur untuk kelas langka; PR-AUC jauh lebih informatif daripada ROC-AUC di sini (Eksperimen C). |
| **Stratified k-fold CV** | Estimasi performa yang stabil — satu split pada data setimpang ini bisa kebetulan tidak mengandung fraud sama sekali di data uji (Eksperimen B). |

**Kesimpulan studi kasus:** pada masalah kelas langka, optimasi harus menargetkan **recall/precision/PR-AUC kelas positif** dengan threshold yang disesuaikan biaya bisnis — bukan memaksimalkan accuracy.

## Contoh Perhitungan Manual dari Confusion Matrix

Diketahui sebuah confusion matrix: **TP = 80, FN = 20, FP = 10, TN = 90** (total 200 data). Hitung manual langkah-demi-langkah:

**1. Accuracy** = (TP + TN) / total = (80 + 90) / 200 = 170 / 200 = **0,85**

**2. Precision** = TP / (TP + FP) = 80 / (80 + 10) = 80 / 90 = **0,8889**

**3. Recall** = TP / (TP + FN) = 80 / (80 + 20) = 80 / 100 = **0,80**

**4. F1-score** = 2 · (P · R) / (P + R) = 2 · (0,8889 · 0,80) / (0,8889 + 0,80) = 2 · 0,7111 / 1,6889 = 1,4222 / 1,6889 = **0,8421**

Sekarang verifikasi dengan scikit-learn:

In [ ]:
# Membangun ulang y_true / y_pred dari angka confusion matrix di atas
y_true = [1]*80 + [1]*20 + [0]*10 + [0]*90   # TP, FN, FP, TN
y_pred = [1]*80 + [0]*20 + [1]*10 + [0]*90

print("Confusion matrix (sklearn):")
print(confusion_matrix(y_true, y_pred))
print("\nAccuracy : %.4f  (manual: 0.8500)" % accuracy_score(y_true, y_pred))
print("Precision: %.4f  (manual: 0.8889)" % precision_score(y_true, y_pred))
print("Recall   : %.4f  (manual: 0.8000)" % recall_score(y_true, y_pred))
print("F1       : %.4f  (manual: 0.8421)" % f1_score(y_true, y_pred))

**Penjelasan output:** confusion matrix yang direkonstruksi tepat menghasilkan [[90, 10], [20, 80]] (baris = aktual 0/1, kolom = prediksi 0/1), dan keempat metrik sklearn (**0,8500 / 0,8889 / 0,8000 / 0,8421**) **persis sama** dengan hitungan manual di atas — membuktikan rumus yang kita pakai sudah benar.

## Latihan Praktikum

### Latihan A — Bandingkan 3 Model dengan Stratified 5-Fold

**Soal:** bandingkan `LogisticRegression`, `KNN (k=5)`, dan `DecisionTree (max_depth=4)` pada dataset breast_cancer memakai stratified 5-fold CV. Sajikan tabel mean ± std untuk accuracy dan F1, lalu simpulkan model terbaik.

In [ ]:
models = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()),
                                    ("lr", LogisticRegression(max_iter=2000, random_state=42))]),
    "KNN (k=5)": Pipeline([("scaler", StandardScaler()),
                           ("knn", KNeighborsClassifier(n_neighbors=5))]),
    "DecisionTree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':<24} {'Accuracy (mean±std)':<20} {'F1 (mean±std)'}")
print("-" * 62)
for nama, m in models.items():
    a = cross_val_score(m, X, y, cv=skf, scoring="accuracy")
    f = cross_val_score(m, X, y, cv=skf, scoring="f1")
    print(f"{nama:<24} {a.mean():.4f} ± {a.std():.4f}      {f.mean():.4f} ± {f.std():.4f}")

**Jawaban & simpulan:**
1. **LogisticRegression terbaik**: accuracy **0,9737 ± 0,0166** dan F1 **0,9794 ± 0,0127** — tertinggi di kedua metrik sekaligus paling stabil (std terkecil).
2. **KNN (k=5)** di posisi kedua: accuracy 0,9631 ± 0,0179, F1 0,9712 ± 0,0138 — hanya ~1 poin di bawah LR, wajar karena data ini memang cukup *linearly separable* setelah scaling.
3. **DecisionTree (depth=4)** paling lemah: accuracy 0,9227 ± 0,0295 — selain rata-ratanya terendah, std-nya **paling besar** (0,0295), artinya performanya paling tidak stabil antar-fold; pohon tunggal memang varians tinggi pada data berfitur banyak.
4. Karena evaluasi memakai **stratified 5-fold**, perbandingan ini adil: semua model diuji pada lipatan yang sama dengan proporsi kelas terjaga, sehingga selisih ~1–5 poin antar model dapat dipercaya bukan sekadar kebetulan split.

### Latihan B — Macro vs Weighted F1 pada Digits: Apa Bedanya?

**Soal:** dari hasil Praktikum 7.3(f) (F1 macro = 0,9619, weighted = 0,9623), jelaskan perbedaan konsep macro vs weighted averaging dan kapan keduanya bisa berbeda jauh.

In [ ]:
# Detail per kelas untuk menjawab Latihan B
from sklearn.metrics import precision_recall_fscore_support
p, r, f1c, sup = precision_recall_fscore_support(yte_d, y_pred_d)
df_lat = pd.DataFrame({"digit": range(10), "F1": np.round(f1c, 4), "support": sup})
print(df_lat.to_string(index=False))
print("\nMacro F1    = rata-rata kolom F1                 = %.4f" % f1c.mean())
print("Weighted F1 = rata-rata F1 dibobot support      = %.4f" % np.average(f1c, weights=sup))
print("Kontribusi digit 8 ke weighted: %.1f%% dari total support"
      % (100 * sup[8] / sup.sum()))

**Jawaban:**
1. **Macro F1 (0,9619)** = rata-rata F1 kesepuluh digit dengan **bobot sama** — digit 8 yang jeblok (F1 = 0,86) "menarik turun" rata-rata sama kuatnya dengan digit 0 yang sempurna. Macro menjawab: *"bagaimana performa model pada tiap kelas secara adil?"*
2. **Weighted F1 (0,9623)** = rata-rata F1 **dibobot jumlah sampel** tiap kelas. Karena support tiap digit hampir sama (43–46), hasilnya nyaris identik dengan macro.
3. Keduanya akan **berbeda jauh** jika: (a) ada kelas minoritas yang performanya buruk — macro akan jauh lebih rendah karena kelas kecil itu tetap dihitung penuh, sedangkan weighted "menenggelamkannya" dalam mayoritas; (b) pada kasus fraud (Studi Kasus), macro/weighted bisa menutupi kegagalan total pada kelas fraud, sehingga yang benar-benar harus dilaporkan adalah **metrik per kelas minoritas itu sendiri**, bukan rata-ratanya.

## Kesimpulan Bab 7

1. **Confusion matrix adalah fondasi evaluasi klasifikasi**: dari empat angka TP/TN/FP/FN diturunkan accuracy, precision, recall, dan F1 — dan *di mana* model salah (FP vs FN) sering lebih penting daripada *berapa banyak* ia salah.
2. **Tidak ada satu metrik yang cocok untuk semua**: accuracy cukup untuk kelas seimbang, tetapi menyesatkan pada kelas langka (Eksperimen C membuktikan model "malas" bisa 84% akurat tanpa menangkap satu pun kasus positif). Pilih metrik sesuai biaya kesalahan: recall saat FN mahal, precision saat FP mahal.
3. **Threshold adalah tuas bisnis, bukan konstanta**: menggeser threshold 0,3 → 0,7 menukar recall sempurna (1,0000) dengan precision tinggi — kurva ROC/PR ada justru untuk menilai model di *semua* threshold, dengan PR-AUC lebih jujur untuk kelas positif yang langka.
4. **Evaluasi yang adil butuh stratified k-fold CV**: satu split bisa kebetulan bagus (0,986 vs 0,9737) dan proporsi kelasnya bisa bergeser; stratified 5-fold memberi estimasi mean ± std yang stabil dan dapat direproduksi.
5. **Pada multiclass, pahami averaging-mu**: macro memperlakukan tiap kelas setara, weighted mengikuti ukuran kelas — keduanya sama hanya jika kelas seimbang; saat timpang, laporkan metrik kelas minoritas secara eksplisit.